# 5. Select Manual mode explicitly

## Before you start

Use [the Ethernet guide](README.md) and the project's Python 3.12 kernel. Each
notebook runs independently against **shutter**, CH **0**, serial **68800777**,
firmware **1.0.7**, KEH3 middle bay at **192.168.1.19:40305**. Port **40303** is
hub management. Keep the hub powered, Ethernet connected and USB IN disconnected.
Close competing owners before connecting. No simulator, scan or USB fallback is used.

Construction is passive. The context releases communication on success or error;
it sends no output command. If release fails, retain `hub` and resolve the reported
error before retrying `hub.close()`. Restart the kernel after source changes.

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

sys.path.insert(0, str(Path("docs/examples") if Path("docs/examples").is_dir() else Path.cwd()))
from shutter_config import SHUTTER  # noqa: E402

from thorlabs_shutter_control import ShutterHub  # noqa: E402

print(SHUTTER.name, SHUTTER.route.endpoint)

The SH05R setup is configured for native KSC101 Manual ON/OFF over Ethernet.
A real mode change first sends OFF/deactivate, then selects Manual and verifies
Manual/OFF. KSC101 remembers the selected mode at power-up; it is not restored
to Trigger automatically.

Before leaving Trigger mode, independently inhibit the optical source and isolate
external trigger control. Only then set `SOURCE_INHIBITED = True` below. The flag
records an operator fact; the software cannot measure the blocked source. Leaving
it False refuses a Trigger->Manual transition. Already Manual is a read-only no-op.
No interlock bypass, trigger configuration, enable or priming command is used.
See [the Python API](../python-api.md) and [the manuals](../sources.md).

In [ ]:
SOURCE_INHIBITED = False  # Set True only after source protection and trigger isolation.
with ShutterHub([SHUTTER]) as hub:
    hub.connect(SHUTTER.name)
    pprint(asdict(hub.set_manual_mode(SHUTTER.name, source_inhibited=SOURCE_INHIBITED).feedback))